# Brain CTA Analysis - Texture Features & PCA

### Contact
- **Alexander R. Webber** (alexander.webber@fda.hhs.gov)
- **Seyed M. Kahaki** (seyed.kahaki@fda.hhs.gov)

---

## Input format

| Item | Details |
|---|---|
| **Root directory** | `experiment_root` - a tree organized as `<root>/<condition>/<case>/ct_simulation/` |
| **CT file** | `MIDA_deformed_<idx>.h5` - dataset key `"reconstruction"`; shape `(Z, Y, X)`, dtype `float32`, values in Hounsfield Units (HU) |
| **Mask file** | `MIDA_deformed_<idx>.mask.h5` - dataset key `"masks/Blood Arteries"`; shape `(Z, Y, X)`, dtype `uint8`, binary `{0, 1}` |
| **Naming convention** | Case folders are `case_<NNNN>`; the zero-padded index `<NNNN>` picks the matching CT/mask filenames |
| **CT preprocessing** | Brain window applied before feature extraction: **center = 40 HU, width = 80 HU** → clipped to `[0, 80]` HU, then normalized to `uint8` `[0, 255]` |
| **Analysis slice** | Middle axial slice (`volume[Z // 2]`) |

Filenames, dataset keys, and the window are all configurable; the values above are the defaults.


## Texture features

Two families are extracted from the masked region of the middle axial slice.

### 1 · Local Binary Pattern (LBP) - histogram

| Parameter | Value |
|---|---|
| Method | `"uniform"` |
| Radius | `3` pixels |
| Sampling points | `24` (`8 × radius`) |
| Output | Normalized histogram over the masked pixels; **26** bins (`n_points + 2`, the number of codes the `uniform` method can produce) |

### 2 · Gray-Level Co-occurrence Matrix (GLCM) - property means

| Parameter | Value |
|---|---|
| Distances | `[1, 3, 5]` pixels |
| Angles | `[0°, 45°, 90°, 135°]` (`[0, π/4, π/2, 3π/4]` radians) |
| Levels | `256` |
| Normalization | Symmetric, normed |
| ROI | Bounding box of the mask on the slice |
| Properties (6) | `contrast`, `dissimilarity`, `homogeneity`, `energy`, `correlation`, `ASM` |
| Output | One mean value per property, averaged over all distance–angle combinations |

**Combined feature vector:** 26 LBP bins + 6 GLCM means → **32 features per case**.


## 1 · Imports


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from IPython.display     import display
from matplotlib          import pyplot as plt

from src.brain_pipeline  import run_cta_analysis


## 2 · Configuration

Defaults to processing all experiments. Can select different experiments by modifying the 'conditions' list. 


In [ ]:
cta_settings = {
    'experiment_root' : '/projects01/didsr-aiml/SDDT/ScoreCard/Brain/Synthetic',

    # Case shown in the three-plane viewer; defaults to the first one found.
    'viewer_condition': '26_07_23_15_21_mA_300',
    'viewer_case'     : 'case_0000',

    # 'conditions'             : ['26_07_23_15_21_mA_300'],  # restrict to some conditions
    # 'max_cases_per_condition': 5,                          # quick trial run
    'mask_dataset'           : 'masks/Blood Arteries',
    # 'window_center'          : 40,
    # 'window_width'           : 80,
    # 'n_components'           : 2,
    # 'save_outputs'           : False,
}


## 3 · Run

Extracting features touches every case in the tree, so this is the slow cell. Set
`max_cases_per_condition` above for a quick trial run first. Outputs are written to
`data/notebook_outputs/brain_cta/`.


In [ ]:
results = run_cta_analysis(**cta_settings)


## 4 · Results

The three-plane viewer confirms the CT window and the mask land where expected before reading
anything into the feature space. The PCA scatter is interactive: click a condition in the legend to
hide it, double-click to isolate it. It is also saved as
`data/notebook_outputs/brain_cta/cta_texture_pca.html`.


In [ ]:
for name, fig in results['figures'].items():
    print(f'--- {name} ---')
    display(fig)
    plt.close(fig)

if results['pca_figure'] is not None:
    results['pca_figure'].show()


In [ ]:
features = results['features']
variance = results['pca']['explained_variance_ratio'] * 100

print(f'Feature table: {features.shape[0]} case(s) × {features.shape[1] - 2} feature(s)')
print(f'PC1 {variance[0]:.1f}% variance, PC2 {variance[1]:.1f}% variance')

if results['skipped']:
    print(f"\n{len(results['skipped'])} case(s) skipped:")

    for condition, case, reason in results['skipped'][:10]:
        print(f'  {condition}/{case} - {reason}')

display(features.head(10))


## 5 · Saved outputs


In [ ]:
for path in results['saved_paths']:
    print(path)
